# 4.6 CNN 過擬合處理

這份 Notebook 使用程式產生的小型方向圖形影像資料集，示範如何觀察 CNN 過擬合，並用 data augmentation、dropout、L2 regularization 與 EarlyStopping 改善。


## 1. 載入套件


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report

tf.keras.utils.set_random_seed(42)


## 2. 建立方向圖形資料集

資料集包含 vertical、horizontal、diagonal 三類 32x32 RGB 圖片。訓練集刻意較小，validation/test 則保留較多變化，用來觀察模型是否只記住訓練樣本。


In [ ]:
CLASS_NAMES = np.array(['vertical', 'horizontal', 'diagonal'])
IMAGE_SIZE = 32

def draw_shape(label, rng, image_size=IMAGE_SIZE, noise=0.12, jitter=3, thickness=2):
    image = rng.normal(loc=0.10, scale=noise, size=(image_size, image_size, 3)).astype('float32')
    color = np.array([0.90, 0.92, 0.88], dtype='float32') + rng.normal(0, 0.03, size=3).astype('float32')
    center = image_size // 2 + int(rng.integers(-jitter, jitter + 1))
    shift = int(rng.integers(-jitter, jitter + 1))

    if label == 0:
        col_start = max(0, center - thickness)
        col_end = min(image_size, center + thickness + 1)
        image[:, col_start:col_end, :] = color
    elif label == 1:
        row_start = max(0, center - thickness)
        row_end = min(image_size, center + thickness + 1)
        image[row_start:row_end, :, :] = color
    else:
        for row in range(image_size):
            col = row + shift
            if 0 <= col < image_size:
                col_start = max(0, col - thickness)
                col_end = min(image_size, col + thickness + 1)
                image[row, col_start:col_end, :] = color

    image += rng.normal(0, noise * 0.6, size=image.shape).astype('float32')
    return np.clip(image, 0.0, 1.0).astype('float32')

def make_shape_dataset(samples_per_class, noise=0.12, jitter=3, thickness=2, seed=42):
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for label in range(len(CLASS_NAMES)):
        for _ in range(samples_per_class):
            images.append(draw_shape(label, rng, noise=noise, jitter=jitter, thickness=thickness))
            labels.append(label)
    images = np.stack(images).astype('float32')
    labels = np.array(labels, dtype='int64')
    index = rng.permutation(len(labels))
    return images[index], labels[index]

x_train, y_train = make_shape_dataset(samples_per_class=12, noise=0.09, jitter=0, thickness=1, seed=1)
x_valid, y_valid = make_shape_dataset(samples_per_class=120, noise=0.22, jitter=10, thickness=1, seed=2)
x_test, y_test = make_shape_dataset(samples_per_class=120, noise=0.22, jitter=10, thickness=1, seed=3)

print('train:', x_train.shape, 'valid:', x_valid.shape, 'test:', x_test.shape)
print('train jitter is intentionally small; validation/test jitter is larger.')
print('classes:', CLASS_NAMES.tolist())


## 3. 觀察範例圖片


In [ ]:
plt.figure(figsize=(8, 3))
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(x_train[i])
    plt.title(CLASS_NAMES[y_train[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()


## 4. 建立容易過擬合的 baseline CNN

這個模型使用較大的 Flatten + Dense 分類頭，參數較多。小資料集上這種模型很容易把訓練資料細節記住。


In [ ]:
def build_baseline_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

baseline_model = build_baseline_model()
baseline_model.summary()


## 5. 訓練 baseline 並觀察 learning curve


In [ ]:
baseline_history = baseline_model.fit(
    x_train, y_train,
    validation_data=(x_valid, y_valid),
    epochs=35,
    batch_size=16,
    verbose=0,
)

def plot_history(history, title):
    hist = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    hist[['accuracy', 'val_accuracy']].plot(ax=axes[0])
    axes[0].set_title(f'{title} accuracy')
    axes[0].set_ylim(0, 1.05)
    hist[['loss', 'val_loss']].plot(ax=axes[1])
    axes[1].set_title(f'{title} loss')
    plt.tight_layout()
    plt.show()

plot_history(baseline_history, 'Baseline CNN')


## 6. 加入 augmentation、dropout、L2 與 EarlyStopping

regularized model 使用較小的分類頭、GlobalAveragePooling、Dropout、L2 regularization 與資料增強，目標是降低訓練集與驗證集之間的落差。


In [ ]:
def build_regularized_model():
    regularizer = tf.keras.regularizers.l2(1e-4)
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomTranslation(0.28, 0.28),
        tf.keras.layers.RandomZoom(0.12),
        tf.keras.layers.RandomContrast(0.15),
    ], name='data_augmentation')

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
        data_augmentation,
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same', kernel_regularizer=regularizer),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same', kernel_regularizer=regularizer),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(96, 3, activation='relu', padding='same', kernel_regularizer=regularizer),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dropout(0.35),
        tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

regularized_model = build_regularized_model()
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=6, restore_best_weights=True
)
regularized_model.summary()


## 7. 訓練 regularized model


In [ ]:
regularized_history = regularized_model.fit(
    x_train, y_train,
    validation_data=(x_valid, y_valid),
    epochs=60,
    batch_size=16,
    callbacks=[early_stop],
    verbose=0,
)
plot_history(regularized_history, 'Regularized CNN')
print('epochs:', len(regularized_history.history['loss']))


## 8. 比較 train、validation、test 表現


In [ ]:
def evaluate_model(name, model):
    rows = []
    for split, x, y in [('train', x_train, y_train), ('valid', x_valid, y_valid), ('test', x_test, y_test)]:
        result = model.evaluate(x, y, verbose=0, return_dict=True)
        rows.append({'model': name, 'split': split, **result})
    return rows

results = pd.DataFrame(evaluate_model('baseline', baseline_model) + evaluate_model('regularized', regularized_model))
results


## 9. 檢查測試集分類結果


In [ ]:
y_prob = regularized_model.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))


## 10. 如何套用自己的資料

換成自己的圖片資料時，先建立 baseline 並畫出 learning curve。若 train 表現明顯高於 validation，再依序加入符合語意的 augmentation、dropout、L2 regularization 與 EarlyStopping。不要一開始只看 test set，test set 應留到模型選擇完成後再使用。


## 11. 小結

CNN 過擬合的重點是先診斷，再處理。Learning curve 可以指出 train/validation 的落差；augmentation、dropout、L2 與 EarlyStopping 則提供改善泛化能力的常用工具。
